# B2-Li 760 r8: дотюн JPEG Q80–95 и сдвиг сетки

Старт из EMA `disentangle_b2_li760_r8_long/ckpt/best.pt`; новый optimizer/scheduler.
3 × 24 000 примеров, LR encoder 1e-5, JPEG/head 3e-5. Архитектура и RGB 760 сохранены.
Обычный sampling по всем доменам, 25% отрицательных; без plain/nonunit focus.

На каждом обучающем примере:
- 70% — без дополнительного JPEG;
- 25% — обычное JPEG-пересохранение Q80–95;
- 5% — обрезка 0–7 пикселей сверху/слева и JPEG-пересохранение Q80–95.

Сдвиг ненулевой хотя бы по одной оси. RGB и GT обрезаются синхронно, без циклического переноса и padding.
DCT и qtable извлекаются из нового JPEG. `[80, 96]` в конфиге означает целые Q80–95.
Все 3 эпохи используют полные кадры после этой малой обрезки, без обычных случайных кропов;
остальные аугментации исходного пайплайна сохраняются.
Development-валидация без аугментаций, выбор best по общей AIC.
После запуска сравнить с исходным раном: plain/nonunit, остальные nonunit по доменам,
unit, Dice положительных и FPR отрицательных. Одного роста среднего AIC недостаточно для вывода о nonunit.

На сервере используйте настройки batch/accumulation из `.env` (например, batch 16, accumulation 1).
Последняя ячейка запускает обучение. Повторный запуск продолжает собственный `last.pt`, если он уже есть.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_r8_jpeg_ft'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
checkpoint = cfg.paths.runs_path / cfg.train.finetune_from
assert checkpoint.is_file(), f'Missing source checkpoint: {checkpoint}'
last = cfg.paths.runs_path / cfg.run_name / 'ckpt' / 'last.pt'
if cfg.train.resume and last.is_file():
    print('Resume this finetune:', last)
else:
    print('Initialize from:', checkpoint, 'weights:', cfg.train.finetune_weights)
a = cfg.augmentation
print(f'No extra JPEG: {1-a.jpeg_recompression_probability:.0%}; '
      f'aligned JPEG: {a.jpeg_recompression_probability-a.jpeg_grid_shift_probability:.0%}; '
      f'shifted JPEG: {a.jpeg_grid_shift_probability:.0%}')
q_min, q_stop = a.jpeg_recompression_quality_range
print(f'JPEG quality: {q_min}..{q_stop-1} inclusive; shifts: 0..7 px, excluding (0, 0)')
print('LR encoder / JPEG / head:', cfg.train.encoder_lr, cfg.train.jpeg_lr, cfg.train.head_lr)
print('Negative fraction:', cfg.train.negative_fraction,
      '; plain/nonunit focus:', cfg.train.plain_nonunit_focus)


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
